# Laboratório 2: O Agente Gerador (RAG Básico)

**Objetivo deste script:** Criar o nosso primeiro funcionário digital. Este agente receberá uma pergunta do usuário, irá até o ChromaDB (a estante mágica que criamos no laboratório anterior) para buscar os trechos mais relevantes do manual e, em seguida, usará um modelo de linguagem (LLM) para formular uma resposta humana baseada *exclusivamente* no que encontrou.

É a fundação da arquitetura RAG (*Retrieval-Augmented Generation*).

In [ ]:
# ==========================================
# PASSO 1: IMPORTAÇÃO DAS FERRAMENTAS
# ==========================================
import os
import warnings

# Limpa mensagens de aviso do terminal para focar apenas no resultado
warnings.filterwarnings("ignore")

# Ferramentas para conectar ao banco de dados que já criamos
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Ferramenta para chamar o nosso cérebro local (Ollama)
from langchain_ollama import OllamaLLM

# Ferramentas para montar a "linha de montagem" do LangChain (LCEL)
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### Reconectando à Memória (VectorDB)
Antes de o agente falar, ele precisa saber como pesquisar. Vamos religar o modelo de tradução matemática (`all-MiniLM-L6-v2`) e apontar o caminho para a pasta onde salvamos o ChromaDB no script anterior.

In [ ]:
# ==========================================
# PASSO 2: CONEXÃO COM O BANCO VETORIAL
# ==========================================

# 1. Caminho relativo da pasta notebooks/ para a pasta data/
DB_PATH = "../data/vectordb"

# 2. Reativar o mesmo tradutor usado na ingestão
print("🔌 Ligando o tradutor de texto para números...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Conectar ao banco de dados existente
print("🗄️ Conectando ao ChromaDB...")
vectordb = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)

# 4. Transformar o banco em um "Recuperador" (Retriever)
# search_kwargs={"k": 3} significa: "Traga apenas os 3 parágrafos mais parecidos com a pergunta"
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("✅ Memória conectada com sucesso!")

### Configurando o "Cérebro" e as Regras (Prompt)
Para garantir que o sistema rode de forma fluida e sem travar a máquina, delegamos o raciocínio para o modelo **Phi-3** via Ollama. Por ser incrivelmente otimizado, ele consegue fazer inferências sofisticadas operando confortavelmente dentro de restrições de hardware, como uma placa gráfica de 2GB de memória (GeForce 940MX).

O `PromptTemplate` é o manual de conduta do agente. É aqui que ordenamos que ele não invente respostas se não encontrar a informação.

In [ ]:
# ==========================================
# PASSO 3: CONFIGURANDO O LLM E O PROMPT
# ==========================================

# 1. Acionar o modelo local (A temperatura=0.1 o torna muito frio, lógico e pouco criativo, ideal para manuais)
print("🧠 Ligando o motor de raciocínio (Phi-3 via Ollama)...")
llm = OllamaLLM(model="phi3", temperature=0.1)

# 2. Escrever a regra de negócio do Agente
template_prompt = """Você é um engenheiro de software especialista em automação RPA.
Sua regra principal de qualidade: Use APENAS os trechos de manuais técnicos fornecidos abaixo para responder à pergunta.
Se a resposta não estiver no manual, diga "Não encontrei a informação no manual técnico fornecido." e não invente comandos.

Manual Técnico recuperado do banco:
{context}

Pergunta do Usuário: {question}

Resposta em português (inclua o código em Python se aplicável):"""

# 3. Empacotar a regra
PROMPT = PromptTemplate.from_template(template_prompt)

# Função auxiliar para pegar os 3 blocos encontrados e colar como um texto único para o LLM ler
def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### A Linha de Montagem (Pipeline LCEL)
O LangChain moderno usa a *LangChain Expression Language* (LCEL). É uma forma visual e profissional de dizer como os dados fluem, usando o símbolo `|` (pipe), que significa "passar o resultado para frente".

O fluxo é:
1. Pega a pergunta (`question`).
2. Usa a pergunta para buscar no banco (`context: retriever | formatar_docs`).
3. Junta a pergunta e os documentos e joga no `PROMPT` (A regra).
4. Envia o Prompt preenchido para o `llm` (O Cérebro).
5. Limpa a saída para um texto puro (`StrOutputParser`).

In [ ]:
# ==========================================
# PASSO 4: MONTANDO O PIPELINE E EXECUTANDO
# ==========================================

# 1. Montando a esteira RAG
rag_chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

# 2. Simulação de uso
pergunta_teste = "Como faço para dar um duplo clique na tela usando o MartonRPA?"

print(f"Usuário: '{pergunta_teste}'\n")
print("🤖 Agente Gerador trabalhando (buscando no banco e redigindo a resposta)...\n")

# 3. Invocar o fluxo
resposta_final = rag_chain.invoke(pergunta_teste)

print("✅ [RESPOSTA DO AGENTE]:")
print("-" * 50)
print(resposta_final)
print("-" * 50)